# 08 - Build and train your own model

**Goals:** run a baseline, compare a controlled change, assess the final model, and adapt to a new CSV.

Bring together the data, training, selection, and inference steps from notebooks 03-07. See the [custom data guide](../guides/03-custom-data.md) when changing datasets. Run cells in order in a fresh kernel; each notebook supplies its own setup.

## Prepare the data

Recreate the training-fitted tensors from notebook 03. Test rows stay reserved for the final assessment in notebook 08.

In [ ]:
from pathlib import Path
import copy
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Keep these small CPU examples quick.


In [ ]:
root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent
data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."

df = pd.read_csv(data_path)

features = ["feature_1", "feature_2", "feature_3"]
train = df[df["split"] == "train"]
validation = df[df["split"] == "validation"]
test = df[df["split"] == "test"]


In [ ]:
scaler = StandardScaler().fit(train[features])  # Fit only on training rows.

def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)


## Choose a baseline

Record the settings before training. Change one setting at a time and compare validation results. Restart from the setup for each experiment; seeds reduce variation but do not guarantee identical results everywhere.

In [ ]:
settings = {
    "seed": 486, "hidden_units": 16, "batch_size": 32,
    "learning_rate": 0.01, "epochs": 20,
}
torch.manual_seed(settings["seed"])
train_data = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_data, batch_size=settings["batch_size"], shuffle=True)
train_eval_loader = DataLoader(train_data, batch_size=64, shuffle=False)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64, shuffle=False)


Keep the two-class logits and cross-entropy pairing. When adapting the data, update both dimensions and the label encoding as needed.

In [ ]:
model = nn.Sequential(
    nn.Linear(len(features), settings["hidden_units"]), nn.ReLU(),
    nn.Linear(settings["hidden_units"], 2),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=settings["learning_rate"])
x_batch, y_batch = next(iter(train_loader))
print("batch:", x_batch.shape, y_batch.shape, x_batch.dtype, y_batch.dtype)
print("model output:", model(x_batch).shape)


## Train and select weights

Use the same epoch and evaluation helpers as notebook 05.

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    for x_batch, y_batch in loader:
        optimizer.zero_grad()  # Each batch needs fresh gradients.
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()


In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            logits = model(x_batch)
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            count += len(y_batch)
    return total_loss / count, correct / count


Within the epoch budget, keep the weights with the lowest validation loss, using the copy-and-restore pattern from notebook 06.

In [ ]:
best_loss, best_epoch = float("inf"), 0
best_state = None
history = []
for epoch in range(1, settings["epochs"] + 1):
    train_one_epoch(model, train_loader)
    train_loss, train_accuracy = evaluate(model, train_eval_loader)
    val_loss, val_accuracy = evaluate(model, val_loader)
    history.append((epoch, train_loss, val_loss, train_accuracy, val_accuracy))
    if val_loss < best_loss:
        best_loss, best_epoch = val_loss, epoch
        best_state = copy.deepcopy(model.state_dict())


In [ ]:
model.load_state_dict(best_state)
model.eval()
val_loss, val_accuracy = evaluate(model, val_loader)
print(f"Selected epoch {best_epoch}: validation loss={val_loss:.3f}, accuracy={val_accuracy:.3f}")


Inspect the learning curves before choosing the next change.

In [ ]:
history_frame = pd.DataFrame(
    history, columns=["epoch", "train_loss", "val_loss", "train_accuracy", "val_accuracy"]
)
history_frame.tail()


In [ ]:
history_frame.plot(x="epoch", y=["train_loss", "val_loss"],
                   ylabel="Cross-entropy loss", title="Learning curves")
plt.show()


## Inspect errors and compare a simple rule

Choose the majority class from training labels, then measure that fixed rule on validation. This avoids choosing the comparison rule from the validation labels.

In [ ]:
majority_class = y_train.bincount().argmax()
majority_accuracy = (y_val == majority_class).float().mean().item()
print("Training majority class:", majority_class.item())
print("Majority-rule validation accuracy:", majority_accuracy)


A confusion matrix shows which classes are mistaken for which. Rows are actual classes and columns are predicted classes, in the order `[0, 1]`.

In [ ]:
with torch.no_grad():
    val_predictions = model(x_val).argmax(dim=1).numpy()
print(confusion_matrix(y_val.numpy(), val_predictions, labels=[0, 1]))


## Record and compare experiments

Add a row after each run, including the selected epoch and validation metrics. Keep a copy of the table outside the kernel or fill in the Markdown table below before restarting.

| Run | Changed setting and reason | Selected epoch | Validation loss | Validation accuracy |
| --- | --- | --- | --- | --- |
| Baseline | Width 16, Adam, learning rate 0.01 | | | |
| Experiment | | | | |

1. State a question, then compare the baseline with at least one controlled change using validation. Consider a regularizer from notebook 06; repeat promising comparisons with several seeds.
2. Select a final approach, rerun it, and then use the test cell below once. Report limitations and what the test estimate represents.
3. For your CSV, change the path, features, label encoding, splits, and model dimensions. Fit preprocessing on training rows and inspect a batch before training.
4. Save the selected model with its architecture and preprocessing as in notebook 07. Record versions, settings, split method, seeds, and whether observations are independent.

In [ ]:
# Add code for your experiment here.


## Final held-out assessment

Enable this only after choosing the final model using validation. Do not use the test result to tune another version; otherwise it becomes part of model selection.

In [ ]:
run_final_test = False  # Change after selecting and rerunning the final approach.

if run_final_test:
    x_test, y_test = tensors(test)
    test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=64, shuffle=False)
    test_loss, test_accuracy = evaluate(model, test_loader)
    print(f"Final test loss: {test_loss:.3f}; test accuracy: {test_accuracy:.3f}")
else:
    print("Test evaluation is reserved until you select a final model.")


## Save and explain your result

Adapt the checkpoint cells from [07](07_saving_and_inference.ipynb) to this architecture. Reload the saved weights and preprocessing and verify a prediction before sharing your result.

In [ ]:
# Save your selected model and verify the reload here.
